# NVIDIA tool-calling detection hunt (FIXED for Jupyter event loop)

This notebook inspects your Redis-backed NVIDIA modelcard registry and lets you re-check tool support via Playwright on-demand.

**Fix included:** no `asyncio.run()` (Jupyter already has a running loop). Use top-level `await`.

**What you’ll get:**
- How many modelcards exist vs. how many models are known
- Breakdown of `tool_supported` (True/False/None)
- Which models are currently missing cards
- A Playwright ground-truth probe for **Tools** button / wrench icon
- Optional: force backfill scraping (if importable)


In [1]:

# If you run this outside the container, set REDIS_URL to the right host/port.
# Example (host machine): export REDIS_URL=redis://localhost:6379/0
import os, json, time, re
from datetime import datetime, timezone

REDIS_URL = "redis://localhost:6381/0"
print("REDIS_URL =", REDIS_URL)


REDIS_URL = redis://localhost:6381/0


In [2]:

import redis
import pandas as pd

r = redis.from_url(REDIS_URL, decode_responses=True)
print("PING:", r.ping())


PING: True


In [3]:

# Registry key patterns — update if you changed prefixes
CARD_PREFIX = "agent:nvidia:modelcard:"
KNOWN_SET   = "agent:nvidia:modelcards:known"

def scan_keys(pattern: str, count: int = 5000):
    out = []
    for k in r.scan_iter(match=pattern, count=count):
        out.append(k)
    return out

known_count = r.scard(KNOWN_SET)
card_keys = scan_keys(CARD_PREFIX + "*")

print("known_count:", known_count)
print("card_keys:", len(card_keys))
print("missing_cards_estimate:", max(0, known_count - len(card_keys)))


known_count: 180
card_keys: 12
missing_cards_estimate: 168


In [4]:

# Load all card records into a DataFrame
def load_cards(keys):
    rows = []
    for k in keys:
        raw = r.get(k)
        if not raw:
            continue
        try:
            obj = json.loads(raw)
        except Exception:
            obj = {"_raw": raw}
        obj["_key"] = k
        rows.append(obj)
    return rows

rows = load_cards(card_keys)
df = pd.DataFrame(rows)

if len(df) == 0:
    print("No modelcard records found.")
else:
    if "validated_at" in df.columns:
        df["validated_at_iso"] = df["validated_at"].apply(
            lambda x: datetime.fromtimestamp(float(x), tz=timezone.utc).isoformat() if pd.notna(x) else None
        )
    if "model_id" not in df.columns and "_key" in df.columns:
        df["model_id"] = df["_key"].str.replace("^" + re.escape(CARD_PREFIX), "", regex=True)

    cols = [c for c in ["model_id","tool_supported","status_code","html_len","validated_at_iso","url","evidence"] if c in df.columns]
    display(df[cols].sort_values(["tool_supported","model_id"], na_position="last").head(80))


,model_id,tool_supported,status_code,html_len,validated_at_iso,url,evidence
9,baai/bge-m3,False,200,0,2026-01-13T16:18:51.830597+00:00,https://build.nvidia.com/baai/bge-m3,"[tools_found=False, tools_disabled=None, wrenc..."
6,baichuan-inc/baichuan2-13b-chat,False,200,0,2026-01-13T16:18:49.039111+00:00,https://build.nvidia.com/baichuan-inc/baichuan...,"[tools_found=False, tools_disabled=None, wrenc..."
5,bigcode/starcoder2-15b,False,200,0,2026-01-13T16:18:50.828220+00:00,https://build.nvidia.com/bigcode/starcoder2-15b,"[tools_found=False, tools_disabled=None, wrenc..."
7,bigcode/starcoder2-7b,False,200,0,2026-01-13T16:19:03.595351+00:00,https://build.nvidia.com/bigcode/starcoder2-7b,"[tools_found=False, tools_disabled=None, wrenc..."
1,bytedance/seed-oss-36b-instruct,False,200,0,2026-01-13T16:19:04.556493+00:00,https://build.nvidia.com/bytedance/seed-oss-36...,"[tools_found=False, tools_disabled=None, wrenc..."
3,databricks/dbrx-instruct,False,200,0,2026-01-13T16:19:03.828958+00:00,https://build.nvidia.com/databricks/dbrx-instruct,"[tools_found=False, tools_disabled=None, wrenc..."
0,01-ai/yi-large,None,200,132801,2026-01-13T14:30:34.842789+00:00,https://build.nvidia.com/01-ai/yi-large,[]
4,abacusai/dracarys-llama-3.1-70b-instruct,None,200,132957,2026-01-13T14:30:34.805979+00:00,https://build.nvidia.com/abacusai/dracarys-lla...,[]
10,adept/fuyu-8b,None,200,132850,2026-01-13T14:30:34.874253+00:00,https://build.nvidia.com/adept/fuyu-8b,[]
2,ai21labs/jamba-1.5-large-instruct,None,200,132970,2026-01-13T14:30:35.572925+00:00,https://build.nvidia.com/ai21labs/jamba-1.5-la...,[]


In [5]:

# Breakdown counts
if len(df):
    print(df["tool_supported"].value_counts(dropna=False))


tool_supported
None     6
False    6
Name: count, dtype: int64


In [6]:

# Which models are still missing a card (in known set but no modelcard key)?
known_ids = list(r.smembers(KNOWN_SET))
known_ids = [x.strip().strip("/").lower() for x in known_ids if x and x.strip()]

have_ids = set()
for k in card_keys:
    mid = k.replace(CARD_PREFIX, "", 1)
    have_ids.add(mid.strip().strip("/").lower())

missing_ids = sorted([mid for mid in known_ids if mid not in have_ids])
print("missing_ids:", len(missing_ids))
print("sample missing:", missing_ids[:30])


missing_ids: 168
sample missing: ['deepseek-ai/deepseek-coder-6.7b-instruct', 'deepseek-ai/deepseek-r1', 'deepseek-ai/deepseek-r1-0528', 'deepseek-ai/deepseek-r1-distill-llama-8b', 'deepseek-ai/deepseek-r1-distill-qwen-14b', 'deepseek-ai/deepseek-r1-distill-qwen-32b', 'deepseek-ai/deepseek-r1-distill-qwen-7b', 'deepseek-ai/deepseek-v3.1', 'deepseek-ai/deepseek-v3.1-terminus', 'deepseek-ai/deepseek-v3.2', 'google/codegemma-1.1-7b', 'google/codegemma-7b', 'google/deplot', 'google/gemma-2-27b-it', 'google/gemma-2-2b-it', 'google/gemma-2-9b-it', 'google/gemma-2b', 'google/gemma-3-12b-it', 'google/gemma-3-1b-it', 'google/gemma-3-27b-it', 'google/gemma-3-4b-it', 'google/gemma-3n-e2b-it', 'google/gemma-3n-e4b-it', 'google/gemma-7b', 'google/paligemma', 'google/recurrentgemma-2b', 'google/shieldgemma-9b', 'gotocompany/gemma-2-9b-cpt-sahabatai-instruct', 'ibm/granite-3.0-3b-a800m-instruct', 'ibm/granite-3.0-8b-instruct']


## On-demand Playwright probe (ground truth)

Static HTML parsing fails on NVIDIA Build because the **Tools** button appears after JS hydration. This probe uses Playwright to verify tool support.


In [7]:

import re
from typing import Dict, Any, List
from playwright.async_api import async_playwright, TimeoutError as PWTimeout

async def probe_tools(model_id: str) -> Dict[str, Any]:
    mid = (model_id or "").strip().strip("/")
    url = f"https://build.nvidia.com/{mid}"
    out: Dict[str, Any] = {
        "model_id": mid,
        "url": url,
        "tools_button_found": False,
        "tools_button_disabled": None,
        "wrench_found": False,
        "wrench_count": 0,
        "tools_button_outer_html": None,
    }

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(viewport={"width": 1400, "height": 900})
        page = await ctx.new_page()

        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        try:
            await page.wait_for_load_state("networkidle", timeout=20000)
        except PWTimeout:
            pass

        tools_btn = page.get_by_role("button", name=re.compile(r"^Tools$", re.I))
        cnt = await tools_btn.count()
        if cnt == 0:
            tools_btn = page.locator('button:has-text("Tools"), [role="button"]:has-text("Tools")')
            cnt = await tools_btn.count()

        if cnt:
            out["tools_button_found"] = True
            try:
                out["tools_button_disabled"] = await tools_btn.first.is_disabled()
            except Exception:
                # fallback via attrs
                attrs = await tools_btn.first.evaluate(
                    '(el) => ({'
                    'disabled: el.hasAttribute("disabled"),'
                    'ariaDisabled: el.getAttribute("aria-disabled"),'
                    'dataDisabled: el.getAttribute("data-disabled"),'
                    'dataState: el.getAttribute("data-state")'
                    '})'
                )
                out["tools_button_disabled"] = bool(
                    attrs.get("disabled")
                    or str(attrs.get("ariaDisabled") or "").lower() == "true"
                    or attrs.get("dataDisabled") is not None
                    or str(attrs.get("dataState") or "").lower() == "disabled"
                )

            out["tools_button_outer_html"] = await tools_btn.first.evaluate("(el) => el.outerHTML")

        wrench = page.locator(
            'svg[data-icon-name="wrench"], '
            'svg[data-src*="wrench.svg"], '
            'use[href^="#wrench_"], '
            'symbol[id^="wrench_"]'
        )
        out["wrench_count"] = await wrench.count()
        out["wrench_found"] = out["wrench_count"] > 0

        await ctx.close()
        await browser.close()

    return out

async def probe_many(ids: List[str]):
    rows = []
    for mid in ids:
        rows.append(await probe_tools(mid))
    return rows


In [8]:

# ✅ JUPYTER-SAFE EXECUTION (no asyncio.run)
tests = ["openai/gpt-oss-120b", "deepseek-ai/deepseek-r1", "deepseek-ai/deepseek-r1-0528"]

rows = await probe_many(tests)
pd.DataFrame(rows)


,model_id,url,tools_button_found,tools_button_disabled,wrench_found,wrench_count,tools_button_outer_html
0,openai/gpt-oss-120b,https://build.nvidia.com/openai/gpt-oss-120b,True,False,True,3,"<button aria-expanded=""false"" data-testid=""nv-..."
1,deepseek-ai/deepseek-r1,https://build.nvidia.com/deepseek-ai/deepseek-r1,False,None,False,0,None
2,deepseek-ai/deepseek-r1-0528,https://build.nvidia.com/deepseek-ai/deepseek-...,True,False,True,3,"<button aria-expanded=""false"" data-testid=""nv-..."


## Optional: force a backfill scrape via your registry class

Run this only if you're executing the notebook **inside** the agent container (or with repo on `PYTHONPATH`).


In [ ]:

# from redis.asyncio import Redis
# from src.agent_service.nvidia_modelcard_registry import NvidiaModelcardRegistry
#
# async def force_scrape(these):
#     redis_async = Redis.from_url(REDIS_URL, decode_responses=True)
#     reg = NvidiaModelcardRegistry(redis_async, max_scrapes_per_run=50, concurrency=3)
#     await reg._scrape_and_store_many([x.lower() for x in these])
#     await redis_async.close()
#
# await force_scrape([
#   "openai/gpt-oss-120b",
#   "openai/gpt-oss-20b",
#   "deepseek-ai/deepseek-r1-0528",
# ])


In [11]:
import sys
import os
import asyncio
import logging
import httpx
import re
from redis.asyncio import Redis

# --- SETUP ---
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

# Initialize Logger
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    force=True
)
log = logging.getLogger("warmup")

REDIS_URL = "redis://localhost:6381/0"

# Imports
try:
    from src.agent_service.nvidia_modelcard_registry import NvidiaModelcardRegistry
    from src.agent_service.config import NVIDIA_BASE_URL, NVIDIA_API_KEY
    from playwright.async_api import TimeoutError as PWTimeout
except ImportError:
    pass

# ==============================================================================
# 1. MONKEY PATCH: Handle "Acknowledge & Continue"
# ==============================================================================
async def _scrape_one_patched(self, model_id: str, sem: asyncio.Semaphore, ctx) -> None:
    mid = self._norm_id(model_id)
    if not mid: return

    if mid in self.HARDCODED_TOOL_SUPPORT:
        await self._store_record(mid, tool_supported=bool(self.HARDCODED_TOOL_SUPPORT[mid]), evidence=["hardcoded"], url=self._build_url(mid), status_code=200, html_len=0)
        log.info(f"[nvidia_cards] {mid} (HARDCODED)")
        return

    existing = await self.redis.get(self._card_key(mid))
    # We skip cache check here to force a retry if you want, 
    # but usually we check if it's already done successfully.
    
    async with sem:
        url = self._build_url(mid)
        page = await ctx.new_page()
        try:
            # 1. Go to page
            await page.goto(url, wait_until="domcontentloaded", timeout=self.nav_timeout_ms)
            
            # --- NEW: POP-UP HANDLING (Specific to your HTML) ---
            try:
                # We look for the exact text "Acknowledge & Continue"
                # OR the generic classes if text varies slightly.
                ack_btn = page.locator("button", has_text="Acknowledge & Continue").first
                
                # If that specific text isn't found, try a broader selector for that button style
                if not await ack_btn.count():
                    ack_btn = page.locator("button.btn-primary.btn-rounded", has_text="Acknowledge").first

                if await ack_btn.is_visible(timeout=3000):
                    log.info(f"[{mid}] Found 'Acknowledge' popup. Clicking...")
                    await ack_btn.click()
                    # Wait a bit for the modal to disappear and page to react
                    await asyncio.sleep(1.0) 
            except Exception as e:
                # It's fine if button is not there
                pass 
            # ----------------------------------------------------

            try:
                await page.wait_for_load_state("networkidle", timeout=self.networkidle_timeout_ms)
            except PWTimeout:
                pass

            # 2. Logic to find Tools button
            tools_btn = page.get_by_role("button", name=re.compile(r"^Tools$", re.I))
            tools_found = (await tools_btn.count()) > 0
            
            tools_disabled = None
            if tools_found:
                try:
                    tools_disabled = await tools_btn.first.is_disabled()
                except: pass

            # 3. Logic to find Wrench
            wrench = page.locator('svg[data-icon-name="wrench"], svg[data-src*="wrench.svg"], use[href^="#wrench_"], symbol[id^="wrench_"]')
            wrench_found = (await wrench.count()) > 0
            
            # 4. Decision
            tool_supported = False
            if tools_found and (tools_disabled is False) and wrench_found:
                tool_supported = True
            
            await self._store_record(
                mid, tool_supported=tool_supported, 
                evidence=[f"tools={tools_found}", f"dis={tools_disabled}", f"wrench={wrench_found}"], 
                url=url, status_code=200, html_len=0
            )
            log.info(f"[nvidia_cards] {mid} supported={tool_supported}")

        except Exception as e:
            log.error(f"[nvidia_cards] {mid} FAILED: {e}")
            if "Target closed" not in str(e) and "Connection closed" not in str(e):
                await self._store_record(mid, tool_supported=None, evidence=[f"err:{str(e)}"], url=url, status_code=0, html_len=0)
        finally:
            try: await page.close()
            except: pass

# APPLY THE PATCH
NvidiaModelcardRegistry._scrape_one = _scrape_one_patched


# ==============================================================================
# 2. MAIN WARMUP SCRIPT
# ==============================================================================
async def main():
    if not NVIDIA_API_KEY:
        log.error("NVIDIA_API_KEY is not set.")
        return

    redis = Redis.from_url(REDIS_URL, decode_responses=True)

    # Concurrency kept low (2) to avoid memory crashes
    registry = NvidiaModelcardRegistry(
        redis,
        validate_interval_seconds=7 * 24 * 3600,
        max_scrapes_per_run=1000, 
        concurrency=2, 
    )

    log.info("Fetching Model List from NVIDIA API...")
    url = f"{NVIDIA_BASE_URL}/models"
    headers = {"Authorization": f"Bearer {NVIDIA_API_KEY}", "Accept": "application/json"}
    
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(url, headers=headers)
            data = resp.json().get("data", [])
        
        model_ids = [m["id"] for m in data if m.get("id")]
        log.info(f"Found {len(model_ids)} models. Starting scrape...")
        
        await registry.update_from_model_list(model_ids)
        log.info("Warmup complete!")

    except Exception as e:
        log.error(f"Warmup fatal error: {e}")
    finally:
        await redis.close()

# Execute
await main()

2026-01-13 22:16:42,622 - warmup - INFO - Fetching Model List from NVIDIA API...
2026-01-13 22:16:42,778 - httpx - INFO - HTTP Request: GET https://integrate.api.nvidia.com/v1/models "HTTP/1.1 200 OK"
2026-01-13 22:16:42,780 - warmup - INFO - Found 182 models. Starting scrape...
2026-01-13 22:16:43,047 - nvidia_modelcard_registry - INFO - [nvidia][registry] ids=182 unique=180 missing=101 due=0 plan=101 cap=1000
2026-01-13 22:17:06,473 - warmup - INFO - [nvidia_cards] microsoft/phi-4-mini-flash-reasoning supported=False
2026-01-13 22:17:06,509 - warmup - INFO - [nvidia_cards] microsoft/phi-3-small-128k-instruct supported=False
2026-01-13 22:17:10,149 - warmup - INFO - [nvidia_cards] microsoft/phi-4-multimodal-instruct supported=False
2026-01-13 22:17:10,267 - warmup - INFO - [nvidia_cards] microsoft/phi-4-mini-instruct supported=False
2026-01-13 22:17:12,712 - warmup - INFO - [nvidia_cards] minimaxai/minimax-m2.1 supported=False
2026-01-13 22:17:17,518 - warmup - INFO - [nvidia_cards] m

CancelledError: 